# vLLM Model Benchmark

Use this after selecting an embedding model. This notebook compares open-source generation models while keeping the embedding fixed.

The default candidate list below is intentionally limited to models whose families are explicitly represented in the current vLLM supported-models docs, so the first sweep is doc-aligned rather than exploratory.

In [ ]:
from __future__ import annotations

import asyncio
import sys
import time
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

repo_root = Path.cwd()
if not (repo_root / "main.py").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "main.py").exists():
            repo_root = parent
            break

sys.path.insert(0, str(repo_root))
load_dotenv(repo_root / ".env")

from eval.model_benchmark import ModelBenchmarkConfig, run_model_benchmark

In [ ]:
GENERATION_MODELS = [
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen3-8B",
    "mistralai/Mistral-7B-Instruct-v0.1",
    "microsoft/Phi-3-mini-4k-instruct",
    "google/gemma-3-1b-it",
]
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
PROVIDER = "vllm"
USE_UMLS = False
SCHEMA_GUIDED = False
SAMPLE_SIZE = 5
NOTE_LIMIT = 25
NOTE_MAX_CHARS = 2000
NOTE_TYPE = "DS"
INPUT_DIR = repo_root / "data" / "evidence" / "mimic_discharge_subset"
MIMIC_CSV = repo_root / "data" / "mimic_iv_note" / "discharge.csv"
OUTPUT_ROOT = repo_root / "output" / "model_benchmark" / time.strftime("%Y%m%dT%H%M%S")

print("generation_models:", GENERATION_MODELS)
print("embedding_model:", EMBEDDING_MODEL)
print("provider:", PROVIDER)
print("output_root:", OUTPUT_ROOT)

In [ ]:
results = asyncio.run(
    run_model_benchmark(
        ModelBenchmarkConfig(
            input_dir=INPUT_DIR,
            output_root=OUTPUT_ROOT,
            generation_models=tuple(GENERATION_MODELS),
            embedding_model=EMBEDDING_MODEL,
            provider=PROVIDER,
            use_umls=USE_UMLS,
            schema_guided=SCHEMA_GUIDED,
            note_limit=NOTE_LIMIT,
            note_max_chars=NOTE_MAX_CHARS,
            note_type=NOTE_TYPE,
            mimic_csv=MIMIC_CSV,
            sample_size=SAMPLE_SIZE,
        )
    )
)

results_df = pd.DataFrame([asdict(result) for result in results])
results_df.sort_values(["exact_match", "mean_query_seconds"], ascending=[False, True], na_position="last")